<a href="https://colab.research.google.com/github/vaibhavjiyer87/engine-nvh-deep-learning/blob/main/notebooks/04_generate_sample_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Create the sample-manifest generation notebook

In [1]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 1 — Mount Google Drive

# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Persistent project location in Google Drive
PROJECT_DRIVE = Path(
    "/content/drive/MyDrive/"
    "NVH_DeepLearning/01_EngineOperatingState"
)

if not PROJECT_DRIVE.exists():
    raise FileNotFoundError(
        f"Project folder was not found:\n{PROJECT_DRIVE}"
    )

print("Google Drive mounted successfully.")
print(f"Project Drive: {PROJECT_DRIVE}")


Mounted at /content/drive
Google Drive mounted successfully.
Project Drive: /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState


In [2]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 2 — Authenticate GitHub + determine Git author identity
# This assumes your GitHub token is stored in Colab Secrets under:
# GITHUB_TOKEN

# ============================================================
# CELL 2 — AUTHENTICATE GITHUB
# ============================================================

from google.colab import userdata

import os
import shutil
import subprocess


# ------------------------------------------------------------
# Repository settings
# ------------------------------------------------------------

REPOSITORY_NAME = "engine-nvh-deep-learning"


# ------------------------------------------------------------
# Load GitHub token securely from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets.\n"
        "Add the token and enable Notebook access."
    )

# GitHub CLI recognizes GH_TOKEN automatically.
os.environ["GH_TOKEN"] = github_token
os.environ["GH_HOST"] = "github.com"


# ------------------------------------------------------------
# Install GitHub CLI if necessary
# ------------------------------------------------------------

if shutil.which("gh") is None:

    print("Installing GitHub CLI...")

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "gh",
        ],
        check=True,
    )


# ------------------------------------------------------------
# Verify authentication
# ------------------------------------------------------------

auth_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".login",
    ],
    capture_output=True,
    text=True,
)

if auth_result.returncode != 0:

    print(auth_result.stderr)

    raise RuntimeError(
        "GitHub authentication failed."
    )


GITHUB_USERNAME = auth_result.stdout.strip()

# ------------------------------------------------------------
# Configure Git commit identity
# ------------------------------------------------------------

user_id_result = subprocess.run(
    [
        "gh",
        "api",
        "user",
        "--jq",
        ".id",
    ],
    capture_output=True,
    text=True,
    check=True,
)

GITHUB_USER_ID = (
    user_id_result.stdout.strip()
)

GIT_NAME = GITHUB_USERNAME

GIT_EMAIL = (
    f"{GITHUB_USER_ID}+"
    f"{GITHUB_USERNAME}@users.noreply.github.com"
)

print("Git identity prepared.")
print(f"Name:  {GIT_NAME}")
print(f"Email: {GIT_EMAIL}")

# ------------------------------------------------------------
# Configure Git to use GitHub CLI authentication
# ------------------------------------------------------------

subprocess.run(
    [
        "gh",
        "auth",
        "setup-git",
        "--hostname",
        "github.com",
        "--force",
    ],
    check=True,
)


print("GitHub authentication successful.")
print(f"GitHub user: {GITHUB_USERNAME}")
print(f"Repository:  {REPOSITORY_NAME}")

Git identity prepared.
Name:  vaibhavjiyer87
Email: 312107027+vaibhavjiyer87@users.noreply.github.com
GitHub authentication successful.
GitHub user: vaibhavjiyer87
Repository:  engine-nvh-deep-learning


In [3]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 3 — Clone repository if .git is missing
# This cell deals with the temporary nature of /content.

# ============================================================
# CELL 3 — RESTORE LOCAL GITHUB REPOSITORY
# ============================================================

from pathlib import Path
import shutil
import subprocess


TEMP_REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

REPOSITORY_IDENTIFIER = (
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}"
)


# ------------------------------------------------------------
# Case 1:
# Valid Git repository already exists
# ------------------------------------------------------------

if (
    TEMP_REPO_DIR.exists()
    and
    (TEMP_REPO_DIR / ".git").exists()
):

    print(
        "Git repository already exists "
        "in this Colab runtime."
    )

    # Pull updates only when the working tree is clean.
    status_result = subprocess.run(
        [
            "git",
            "-C",
            str(TEMP_REPO_DIR),
            "status",
            "--porcelain",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    if status_result.stdout.strip():

        print(
            "Local changes detected."
        )

        print(
            "Automatic git pull skipped "
            "to avoid overwriting local work."
        )

    else:

        print(
            "Working tree is clean. "
            "Updating from GitHub..."
        )

        pull_result = subprocess.run(
            [
                "git",
                "-C",
                str(TEMP_REPO_DIR),
                "pull",
                "--ff-only",
            ],
            capture_output=True,
            text=True,
        )

        print(pull_result.stdout)

        if pull_result.returncode != 0:
            print(pull_result.stderr)


# ------------------------------------------------------------
# Case 2:
# Folder exists, but it is NOT a Git repository
# ------------------------------------------------------------

elif TEMP_REPO_DIR.exists():

    raise RuntimeError(
        f"The folder exists but is not a Git repository:\n"
        f"{TEMP_REPO_DIR}\n\n"
        "Do not run git init. Inspect or back up the folder "
        "before removing it and rerunning this cell."
    )


# ------------------------------------------------------------
# Case 3:
# Fresh runtime — clone repository
# ------------------------------------------------------------

else:

    print(
        "Repository not present in this runtime."
    )

    print(
        f"Cloning {REPOSITORY_IDENTIFIER}..."
    )

    clone_result = subprocess.run(
        [
            "gh",
            "repo",
            "clone",
            REPOSITORY_IDENTIFIER,
            str(TEMP_REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

    print(clone_result.stdout)

    if clone_result.returncode != 0:

        print(clone_result.stderr)

        raise RuntimeError(
            "Repository clone failed."
        )


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

if not (
    TEMP_REPO_DIR
    / ".git"
).exists():

    raise RuntimeError(
        "Repository restoration failed."
    )


print("Local Git repository is ready.")
print(f"Location: {TEMP_REPO_DIR}")

Repository not present in this runtime.
Cloning vaibhavjiyer87/engine-nvh-deep-learning...

Local Git repository is ready.
Location: /content/engine-nvh-deep-learning


In [18]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 4 — Define REPO_DIR and all standard paths + apply Git author identity

# ============================================================
# CELL 4 — DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# GitHub working repository
# ------------------------------------------------------------

REPO_DIR = (
    Path("/content")
    / REPOSITORY_NAME
)

if not (
    REPO_DIR
    / ".git"
).exists():

    raise FileNotFoundError(
        f"Valid Git repository not found at:\n"
        f"{REPO_DIR}\n\n"
        "Run Cell 3 first."
    )


# ------------------------------------------------------------
# Persistent raw dataset
# ------------------------------------------------------------

RAW_ROOT = (
    PROJECT_DRIVE
    / "data"
    / "raw"
    / "procedural_engine_sounds"
)

DATASET_ROOT = (
    RAW_ROOT
    / "dataset"
)

AUDIO_DIR = (
    DATASET_ROOT
    / "audio"
    / "A_full_set"
)


# ------------------------------------------------------------
# Persistent Google Drive manifests
# ------------------------------------------------------------

DRIVE_MANIFEST_DIR = (
    PROJECT_DRIVE
    / "data"
    / "manifests"
)


# ------------------------------------------------------------
# GitHub configuration
# ------------------------------------------------------------

CONFIG_DIR = (
    REPO_DIR
    / "configs"
)

REPORT_DIR = (
    REPO_DIR
    / "reports"
)

SPLIT_DIR = (
    REPO_DIR
    / "data"
    / "splits"
)


# ------------------------------------------------------------
# GitHub result directories
# ------------------------------------------------------------

RESULTS_DIR = (
    REPO_DIR
    / "results"
)

FIGURE_DIR = (
    RESULTS_DIR
    / "figures"
)

TABLE_DIR = (
    RESULTS_DIR
    / "tables"
)

DATA_AUDIT_FIGURE_DIR = (
    FIGURE_DIR
    / "data_audit"
)

DESIGN_FIGURE_DIR = (
    FIGURE_DIR
    / "preprocessing_design"
)

SPLIT_FIGURE_DIR = (
    FIGURE_DIR
    / "split_design"
)


# ------------------------------------------------------------
# Persistent Drive output directories
# ------------------------------------------------------------

DRIVE_OUTPUT_DIR = (
    PROJECT_DRIVE
    / "outputs"
)

DRIVE_DESIGN_TABLE_DIR = (
    DRIVE_OUTPUT_DIR
    / "tables"
    / "preprocessing_design"
)


# ------------------------------------------------------------
# Create output folders if missing
# ------------------------------------------------------------

directories_to_create = [
    CONFIG_DIR,
    REPORT_DIR,
    SPLIT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    DATA_AUDIT_FIGURE_DIR,
    DESIGN_FIGURE_DIR,
    SPLIT_FIGURE_DIR,
    DRIVE_DESIGN_TABLE_DIR,
]

for directory in directories_to_create:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("Project paths restored.")
print()
print(f"REPO_DIR:     {REPO_DIR}")
print(f"PROJECT_DRIVE:{PROJECT_DRIVE}")
print(f"RAW_ROOT:     {RAW_ROOT}")
print(f"AUDIO_DIR:    {AUDIO_DIR}")

# ------------------------------------------------------------
# Apply Git commit identity to cloned repository
# ------------------------------------------------------------

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.name",
        GIT_NAME,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "config",
        "user.email",
        GIT_EMAIL,
    ],
    check=True,
)

print("Git commit identity applied to repository.")

# ------------------------------------------------------------
# Restore project Python dependencies
# ------------------------------------------------------------

import subprocess

REQUIREMENTS_PATH = (
    REPO_DIR
    / "requirements.txt"
)

if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(
        f"requirements.txt not found:\n"
        f"{REQUIREMENTS_PATH}"
    )

install_result = subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    capture_output=True,
    text=True,
)

if install_result.returncode != 0:

    print(install_result.stdout)
    print(install_result.stderr)

    raise RuntimeError(
        "Project dependency installation failed."
    )

print(
    "Project Python dependencies restored."
)

Project paths restored.

REPO_DIR:     /content/engine-nvh-deep-learning
PROJECT_DRIVE:/content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState
RAW_ROOT:     /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds
AUDIO_DIR:    /content/drive/MyDrive/NVH_DeepLearning/01_EngineOperatingState/data/raw/procedural_engine_sounds/dataset/audio/A_full_set
Git commit identity applied to repository.
Project Python dependencies restored.


In [22]:
# A 'fresh Colab runtime' cells to be run (total 5)

# Cell 5 — Load persistent analysis data
# This cell restores the major tables you've created so far.
# This is conditional, as some files may not exist yet depending on
# where you are in the project.

# ============================================================
# CELL 5 — LOAD PERSISTENT ANALYSIS DATA
# ============================================================

# ============================================================
# STANDARD PROJECT IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import yaml

from tqdm.auto import tqdm

print(
    "Standard project libraries imported."
)


# ------------------------------------------------------------
# 1. Raw-file manifest — REQUIRED
# ------------------------------------------------------------

RAW_MANIFEST_PATH = (
    DRIVE_MANIFEST_DIR
    / "raw_file_manifest_v001.csv"
)

if not RAW_MANIFEST_PATH.exists():

    raise FileNotFoundError(
        f"Required raw manifest not found:\n"
        f"{RAW_MANIFEST_PATH}"
    )


raw_manifest = pd.read_csv(
    RAW_MANIFEST_PATH
)

print(
    f"LOADED raw_manifest: "
    f"{raw_manifest.shape}"
)


# ------------------------------------------------------------
# 2. PREP-001 detailed target analysis — OPTIONAL
# ------------------------------------------------------------

PREP001_TARGET_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "prep001_target_analysis_v001.csv.gz"
)

if PREP001_TARGET_PATH.exists():

    prep001_targets = pd.read_csv(
        PREP001_TARGET_PATH
    )

    print(
        f"LOADED prep001_targets: "
        f"{prep001_targets.shape}"
    )

else:

    prep001_targets = None

    print(
        "NOT FOUND: prep001 target analysis "
        "(this is okay if it has not been generated yet)."
    )


# ------------------------------------------------------------
# 3. Window candidate metrics — OPTIONAL
# ------------------------------------------------------------

WINDOW_CANDIDATE_PATH = (
    DRIVE_DESIGN_TABLE_DIR
    / "window_candidate_metrics_v001.csv.gz"
)

if WINDOW_CANDIDATE_PATH.exists():

    window_candidates = pd.read_csv(
        WINDOW_CANDIDATE_PATH
    )

    print(
        f"LOADED window_candidates: "
        f"{window_candidates.shape}"
    )

else:

    window_candidates = None

    print(
        "NOT FOUND: window candidate metrics."
    )


# ------------------------------------------------------------
# 4. Frozen SPLIT-001 file assignment — OPTIONAL
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if FILE_SPLIT_PATH.exists():

    file_split = pd.read_csv(
        FILE_SPLIT_PATH
    )

    print(
        f"LOADED file_split: "
        f"{file_split.shape}"
    )

else:

    file_split = None

    print(
        "NOT FOUND: SPLIT-001 file assignment."
    )


# ============================================================
# LOAD REQUIRED FROZEN PROJECT SPECIFICATIONS
# ============================================================

import yaml


# ------------------------------------------------------------
# PREP-001 — REQUIRED
# ------------------------------------------------------------

PREPROCESSING_CONFIG_PATH = (
    CONFIG_DIR
    / "preprocessing_v001.yaml"
)

if not PREPROCESSING_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required PREP-001 configuration is missing:\n"
        f"{PREPROCESSING_CONFIG_PATH}"
    )


try:

    with open(
        PREPROCESSING_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        preprocessing_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "PREP-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
    != "PREP-001"
):

    raise RuntimeError(
        "Unexpected preprocessing version."
    )


print(
    "LOADED preprocessing_config: PREP-001"
)


# ------------------------------------------------------------
# SPLIT-001 — REQUIRED
# ------------------------------------------------------------

SPLIT_CONFIG_PATH = (
    CONFIG_DIR
    / "split_v001.yaml"
)

if not SPLIT_CONFIG_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 configuration is missing:\n"
        f"{SPLIT_CONFIG_PATH}"
    )


try:

    with open(
        SPLIT_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        split_config = (
            yaml.safe_load(file)
        )

except yaml.YAMLError as error:

    raise RuntimeError(
        "SPLIT-001 exists but is invalid YAML.\n"
        f"{error}"
    )


if (
    split_config[
        "specification"
    ]["split_version"]
    != "SPLIT-001"
):

    raise RuntimeError(
        "Unexpected split version."
    )


print(
    "LOADED split_config: SPLIT-001"
)


# ------------------------------------------------------------
# FILE SPLIT — REQUIRED
# ------------------------------------------------------------

FILE_SPLIT_PATH = (
    SPLIT_DIR
    / "file_split_v001.csv"
)

if not FILE_SPLIT_PATH.exists():

    raise FileNotFoundError(
        "Required SPLIT-001 assignment file is missing:\n"
        f"{FILE_SPLIT_PATH}"
    )


file_split = pd.read_csv(
    FILE_SPLIT_PATH
)

print(
    f"LOADED file_split: "
    f"{file_split.shape}"
)


# ============================================================
# DERIVE FROZEN RUNTIME CONSTANTS
# ============================================================

# ------------------------------------------------------------
# PREP-001 identity
# ------------------------------------------------------------

PREPROCESSING_VERSION = (
    preprocessing_config[
        "specification"
    ]["preprocessing_version"]
)

DATASET_SUBSET = (
    preprocessing_config[
        "specification"
    ]["dataset_subset"]
)


# ------------------------------------------------------------
# Windowing
# ------------------------------------------------------------

WINDOW_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["window_duration_s"]
)

OVERLAP_FRACTION = float(
    preprocessing_config[
        "windowing"
    ]["overlap_fraction"]
)

HOP_DURATION_S = float(
    preprocessing_config[
        "windowing"
    ]["hop_duration_s"]
)


# ------------------------------------------------------------
# Source-data definition
# ------------------------------------------------------------

SOURCE_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "source_data"
    ]["source_sample_rate_hz"]
)

EXPECTED_CHANNELS = int(
    preprocessing_config[
        "source_data"
    ]["expected_channels"]
)

RPM_SCALE_FACTOR = float(
    preprocessing_config[
        "source_data"
    ]["rpm_scale_factor"]
)

TORQUE_SCALE_FACTOR_NM = float(
    preprocessing_config[
        "source_data"
    ]["torque_scale_factor_nm"]
)


# ------------------------------------------------------------
# YAML channel numbers are human-readable 1-based numbers.
# NumPy arrays use 0-based indexing.
# ------------------------------------------------------------

RPM_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["rpm"]
    )
    - 1
)

TORQUE_CHANNEL_INDEX = (
    int(
        preprocessing_config[
            "source_data"
        ]["annotation_channels"]["torque"]
    )
    - 1
)


# ------------------------------------------------------------
# Audio processing
# ------------------------------------------------------------

TARGET_SAMPLE_RATE_HZ = int(
    preprocessing_config[
        "audio_processing"
    ]["target_sample_rate_hz"]
)

AUDIO_CHANNEL_STRATEGY = (
    preprocessing_config[
        "audio_processing"
    ]["channel_strategy"]
)


# ------------------------------------------------------------
# Steady-state criterion
# ------------------------------------------------------------

ABSOLUTE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_absolute_range_rpm"
    ]
)

RELATIVE_RPM_LIMIT = float(
    preprocessing_config[
        "operating_state"
    ]["steady_state_rule"][
        "maximum_relative_range_fraction"
    ]
)


# ------------------------------------------------------------
# SPLIT-001 identity
# ------------------------------------------------------------

SPLIT_VERSION = (
    split_config[
        "specification"
    ]["split_version"]
)


print("Frozen runtime constants restored.")
print()
print(f"Preprocessing:       {PREPROCESSING_VERSION}")
print(f"Split:               {SPLIT_VERSION}")
print(f"Dataset subset:      {DATASET_SUBSET}")
print(f"Window duration:     {WINDOW_DURATION_S} s")
print(f"Overlap:             {100 * OVERLAP_FRACTION:.0f}%")
print(f"Source sample rate:  {SOURCE_SAMPLE_RATE_HZ} Hz")
print(f"Target sample rate:  {TARGET_SAMPLE_RATE_HZ} Hz")
print(f"Expected channels:   {EXPECTED_CHANNELS}")
print(f"RPM channel index:   {RPM_CHANNEL_INDEX}")
print(f"Torque channel idx:  {TORQUE_CHANNEL_INDEX}")
print(f"Channel strategy:    {AUDIO_CHANNEL_STRATEGY}")

# ============================================================
# VALIDATE FROZEN RUNTIME CONSTANTS
# ============================================================

assert PREPROCESSING_VERSION == "PREP-001", (
    f"Expected PREP-001, found {PREPROCESSING_VERSION}"
)

assert SPLIT_VERSION == "SPLIT-001", (
    f"Expected SPLIT-001, found {SPLIT_VERSION}"
)

assert SOURCE_SAMPLE_RATE_HZ == 48000, (
    "Unexpected PREP-001 source sample rate."
)

assert TARGET_SAMPLE_RATE_HZ == 16000, (
    "Unexpected PREP-001 target sample rate."
)

assert EXPECTED_CHANNELS == 4, (
    "Unexpected PREP-001 channel count."
)

assert WINDOW_DURATION_S > 0

assert 0 <= OVERLAP_FRACTION < 1

assert RPM_CHANNEL_INDEX < EXPECTED_CHANNELS

assert TORQUE_CHANNEL_INDEX < EXPECTED_CHANNELS

assert file_split["file_id"].is_unique

assert set(
    file_split["split"].unique()
) == {
    "train",
    "validation",
    "test",
}

print(
    "PASS: Frozen runtime constants validated."
)

# ============================================================
# PROJECT SESSION READINESS SUMMARY
# ============================================================

print()
print("=" * 64)
print("NVH DEEP-LEARNING PROJECT SESSION READY")
print("=" * 64)

print(
    f"Raw manifest:       "
    f"{len(raw_manifest):,} source files"
)

print(
    f"Preprocessing:      "
    f"{PREPROCESSING_VERSION}"
)

print(
    f"Dataset split:      "
    f"{SPLIT_VERSION}"
)

print(
    f"Window:             "
    f"{WINDOW_DURATION_S:.1f} s"
)

print(
    f"Overlap:            "
    f"{100 * OVERLAP_FRACTION:.0f}%"
)

print(
    f"Source sample rate: "
    f"{SOURCE_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Model sample rate:  "
    f"{TARGET_SAMPLE_RATE_HZ:,} Hz"
)

print(
    f"Audio strategy:     "
    f"{AUDIO_CHANNEL_STRATEGY}"
)

print()
print("SPLIT-001:")

print(
    file_split[
        "split"
    ]
    .value_counts()
    .to_string()
)

if (
    "sample_manifest" in globals()
    and
    sample_manifest is not None
):

    print()
    print(
        f"Sample manifest:    "
        f"{len(sample_manifest):,} samples"
    )

else:

    print()
    print(
        "Sample manifest:    "
        "not generated yet"
    )

print("=" * 64)

Standard project libraries imported.
LOADED raw_manifest: (767, 21)
LOADED prep001_targets: (16503, 23)
LOADED window_candidates: (58653, 16)
LOADED file_split: (767, 11)
LOADED preprocessing_config: PREP-001
LOADED split_config: SPLIT-001
LOADED file_split: (767, 11)
Frozen runtime constants restored.

Preprocessing:       PREP-001
Split:               SPLIT-001
Dataset subset:      A_full_set
Window duration:     1.0 s
Overlap:             50%
Source sample rate:  48000 Hz
Target sample rate:  16000 Hz
Expected channels:   4
RPM channel index:   2
Torque channel idx:  3
Channel strategy:    mono_average
PASS: Frozen runtime constants validated.

SESSION RESTORE COMPLETE
Raw files in manifest: 767
PREP-001 windows: 16503

SPLIT-001 allocation:
split
train         536
test          116
validation    115
Name: count, dtype: int64


In [23]:
# Step 1 — Create the sample-manifest generation notebook
# Cell-1 Verify sample-manifest pre-requisites

required_runtime_objects = [
    "preprocessing_config",
    "split_config",
    "file_split",
    "PREPROCESSING_VERSION",
    "SPLIT_VERSION",
    "WINDOW_DURATION_S",
    "OVERLAP_FRACTION",
    "SOURCE_SAMPLE_RATE_HZ",
    "TARGET_SAMPLE_RATE_HZ",
    "EXPECTED_CHANNELS",
    "RPM_SCALE_FACTOR",
    "TORQUE_SCALE_FACTOR_NM",
    "RPM_CHANNEL_INDEX",
    "TORQUE_CHANNEL_INDEX",
    "AUDIO_CHANNEL_STRATEGY",
    "ABSOLUTE_RPM_LIMIT",
    "RELATIVE_RPM_LIMIT",
]

missing_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required runtime objects are missing:\n"
        + "\n".join(
            f" - {name}"
            for name in missing_objects
        )
        + "\n\nRun Startup Cells 1–5 first."
    )


assert PREPROCESSING_VERSION == "PREP-001"
assert SPLIT_VERSION == "SPLIT-001"

print(
    "PASS: All SAMPLE-MANIFEST-001 "
    "prerequisites are available."
)

PASS: All SAMPLE-MANIFEST-001 prerequisites are available.


In [25]:
# Step 2 — Generate one manifest row per model window
# This is the main calculation.
# For every source WAV in file_split_v001.csv, we will:
  # Read the four-channel WAV.
  # Recover RPM and torque.
  # Cut it logically into PREP-001 windows.
  # Calculate RPM and torque statistics.
  # Determine steady/transient state.
  # Inherit the parent's SPLIT-001 assignment.
  # Generate a unique sample_id.

# Cell 2 — Generate the manifest

# ============================================================
# GENERATE SAMPLE_MANIFEST_V001
# ============================================================

sample_records = []
generation_errors = []


for row in tqdm(
    file_split.itertuples(index=False),
    total=len(file_split),
    desc="Generating sample manifest",
):

    wav_path = (
        RAW_ROOT
        / row.relative_file_path
    )

    # --------------------------------------------------------
    # Validate source file
    # --------------------------------------------------------

    if not wav_path.exists():

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": "Source WAV does not exist",
            }
        )

        continue


    try:

        signal, sample_rate = sf.read(
            wav_path,
            always_2d=True,
        )

    except Exception as error:

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": str(error),
            }
        )

        continue


    # --------------------------------------------------------
    # Validate WAV structure
    # --------------------------------------------------------

    if sample_rate != SOURCE_SAMPLE_RATE_HZ:

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": (
                    f"Unexpected sample rate "
                    f"{sample_rate}"
                ),
            }
        )

        continue


    if signal.shape[1] != EXPECTED_CHANNELS:

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": (
                    f"Unexpected channel count "
                    f"{signal.shape[1]}"
                ),
            }
        )

        continue


    if not np.isfinite(signal).all():

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": (
                    "Non-finite values detected"
                ),
            }
        )

        continue


    # --------------------------------------------------------
    # Recover physical annotation traces
    # --------------------------------------------------------

    rpm_trace = (
        signal[:, RPM_CHANNEL_INDEX]
        * RPM_SCALE_FACTOR
    )

    torque_trace_nm = (
        signal[:, TORQUE_CHANNEL_INDEX]
        * TORQUE_SCALE_FACTOR_NM
    )


    # --------------------------------------------------------
    # PREP-001 window sizes
    # --------------------------------------------------------

    window_samples = int(
        round(
            WINDOW_DURATION_S
            * sample_rate
        )
    )

    hop_samples = int(
        round(
            window_samples
            * (1 - OVERLAP_FRACTION)
        )
    )


    if len(signal) < window_samples:

        generation_errors.append(
            {
                "file_id": row.file_id,
                "error": (
                    "Source WAV shorter than "
                    "one PREP-001 window"
                ),
            }
        )

        continue


    number_of_windows = (
        1
        +
        (
            len(signal)
            - window_samples
        )
        // hop_samples
    )


    # --------------------------------------------------------
    # Process each logical window
    # --------------------------------------------------------

    for window_index in range(
        number_of_windows
    ):

        start_sample = (
            window_index
            * hop_samples
        )

        end_sample = (
            start_sample
            + window_samples
        )


        rpm_window = rpm_trace[
            start_sample:end_sample
        ]

        torque_window = torque_trace_nm[
            start_sample:end_sample
        ]


        # ----------------------------------------------------
        # RPM statistics
        # ----------------------------------------------------

        rpm_mean = float(
            np.mean(rpm_window)
        )

        rpm_median = float(
            np.median(rpm_window)
        )

        rpm_min = float(
            np.min(rpm_window)
        )

        rpm_max = float(
            np.max(rpm_window)
        )

        rpm_std = float(
            np.std(rpm_window)
        )

        rpm_range = (
            rpm_max
            - rpm_min
        )


        # ----------------------------------------------------
        # Torque statistics
        # ----------------------------------------------------

        torque_mean_nm = float(
            np.mean(torque_window)
        )

        torque_median_nm = float(
            np.median(torque_window)
        )

        torque_min_nm = float(
            np.min(torque_window)
        )

        torque_max_nm = float(
            np.max(torque_window)
        )

        torque_std_nm = float(
            np.std(torque_window)
        )

        torque_range_nm = (
            torque_max_nm
            - torque_min_nm
        )


        # ----------------------------------------------------
        # PREP-001 steady-state rule
        # ----------------------------------------------------

        allowed_rpm_range = max(
            ABSOLUTE_RPM_LIMIT,
            RELATIVE_RPM_LIMIT
            * abs(rpm_mean),
        )

        is_steady = (
            rpm_range
            <= allowed_rpm_range
        )


        operating_state = (
            "steady"
            if is_steady
            else "transient"
        )


        # ----------------------------------------------------
        # Unique stable sample ID
        # ----------------------------------------------------

        sample_id = (
            f"{row.file_id}"
            f"__W{window_index:05d}"
        )


        # ----------------------------------------------------
        # Build manifest row
        # ----------------------------------------------------

        sample_records.append(
            {
                "sample_id":
                    sample_id,

                "parent_file_id":
                    row.file_id,

                "subset_id":
                    row.subset_id,

                "relative_raw_path":
                    row.relative_file_path,

                "window_index":
                    window_index,

                "source_start_sample":
                    start_sample,

                "source_end_sample_exclusive":
                    end_sample,

                "start_time_s":
                    start_sample
                    / sample_rate,

                "end_time_s":
                    end_sample
                    / sample_rate,

                "window_duration_s":
                    WINDOW_DURATION_S,

                "source_sample_rate_hz":
                    sample_rate,

                "target_sample_rate_hz":
                    TARGET_SAMPLE_RATE_HZ,

                "source_window_num_samples":
                    window_samples,

                "expected_model_num_samples":
                    int(
                        round(
                            WINDOW_DURATION_S
                            * TARGET_SAMPLE_RATE_HZ
                        )
                    ),

                "audio_channel_strategy":
                    AUDIO_CHANNEL_STRATEGY,

                "rpm_mean":
                    rpm_mean,

                "rpm_median":
                    rpm_median,

                "rpm_min":
                    rpm_min,

                "rpm_max":
                    rpm_max,

                "rpm_std":
                    rpm_std,

                "rpm_range":
                    rpm_range,

                "allowed_rpm_range":
                    allowed_rpm_range,

                "torque_mean_nm":
                    torque_mean_nm,

                "torque_median_nm":
                    torque_median_nm,

                "torque_min_nm":
                    torque_min_nm,

                "torque_max_nm":
                    torque_max_nm,

                "torque_std_nm":
                    torque_std_nm,

                "torque_range_nm":
                    torque_range_nm,

                "operating_state":
                    operating_state,

                "model_eligible_rpm_v001":
                    bool(is_steady),

                "quality_flag":
                    "pass",

                "split":
                    row.split,

                "preprocessing_version":
                    PREPROCESSING_VERSION,

                "split_version":
                    SPLIT_VERSION,
            }
        )


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

sample_manifest = pd.DataFrame(
    sample_records
)


print()
print(
    f"Generated samples: "
    f"{len(sample_manifest):,}"
)

print(
    f"Generation errors: "
    f"{len(generation_errors)}"
)

display(
    sample_manifest.head()
)

Generating sample manifest:   0%|          | 0/767 [00:00<?, ?it/s]


Generated samples: 16,503
Generation errors: 0


,sample_id,parent_file_id,subset_id,relative_raw_path,window_index,source_start_sample,source_end_sample_exclusive,start_time_s,end_time_s,window_duration_s,...,torque_min_nm,torque_max_nm,torque_std_nm,torque_range_nm,operating_state,model_eligible_rpm_v001,quality_flag,split,preprocessing_version,split_version
0,A-0000__W00000,A-0000,A_full_set,dataset/audio/A_full_set/001_Engine-A.wav,0,0,48000,0.0,1.0,1.0,...,-26.519775,-22.918701,0.939696,3.601074,steady,True,pass,train,PREP-001,SPLIT-001
1,A-0000__W00001,A-0000,A_full_set,dataset/audio/A_full_set/001_Engine-A.wav,1,24000,72000,0.5,1.5,1.0,...,-25.115967,-20.874023,0.964778,4.241943,transient,False,pass,train,PREP-001,SPLIT-001
2,A-0000__W00002,A-0000,A_full_set,dataset/audio/A_full_set/001_Engine-A.wav,2,48000,96000,1.0,2.0,1.0,...,-23.254395,-19.775391,1.038085,3.479004,transient,False,pass,train,PREP-001,SPLIT-001
3,A-0000__W00003,A-0000,A_full_set,dataset/audio/A_full_set/001_Engine-A.wav,3,72000,120000,1.5,2.5,1.0,...,-21.514893,-19.287109,0.551702,2.227783,steady,True,pass,train,PREP-001,SPLIT-001
4,A-0000__W00004,A-0000,A_full_set,dataset/audio/A_full_set/001_Engine-A.wav,4,96000,144000,2.0,3.0,1.0,...,-20.751953,-17.364502,0.825930,3.387451,steady,True,pass,train,PREP-001,SPLIT-001


In [31]:
# Step 3 — Validate the manifest before saving it
# Do not save/freeze the manifest immediately.
# First prove that it satisfies PREP-001 and SPLIT-001.

# 3A. Make sure generation had no errors
# ============================================================
# CHECK GENERATION ERRORS
# ============================================================

if generation_errors:

    generation_error_df = pd.DataFrame(
        generation_errors
    )

    display(generation_error_df)

    raise RuntimeError(
        f"{len(generation_errors)} source files "
        "could not be processed. "
        "Resolve these before freezing the manifest."
    )

else:

    print(
        "PASS: All SPLIT-001 source files "
        "were processed successfully."
    )

#3B. Check sample IDs
assert (
    sample_manifest[
        "sample_id"
    ].is_unique
)

assert (
    sample_manifest[
        "sample_id"
    ].notna().all()
)

print(
    "PASS: Every model sample "
    "has a unique sample_id."
)

# 3C. Confirm every parent file is represented
expected_parent_ids = set(
    file_split["file_id"]
)

manifest_parent_ids = set(
    sample_manifest[
        "parent_file_id"
    ]
)

missing_parent_ids = (
    expected_parent_ids
    - manifest_parent_ids
)

unexpected_parent_ids = (
    manifest_parent_ids
    - expected_parent_ids
)


assert not missing_parent_ids, (
    f"Missing parents: "
    f"{missing_parent_ids}"
)

assert not unexpected_parent_ids, (
    f"Unexpected parents: "
    f"{unexpected_parent_ids}"
)


print(
    "PASS: Every SPLIT-001 source WAV "
    "generated at least one sample."
)


# 3D. Critical leakage check
# Every parent WAV must appear in exactly one split.
parent_split_counts = (
    sample_manifest
    .groupby(
        "parent_file_id"
    )["split"]
    .nunique()
)

assert (
    parent_split_counts.max()
    == 1
)

print(
    "PASS: No parent WAV spans "
    "multiple dataset splits."
)

# also compare against the original SPLIT-001 assignments:
expected_split_map = (
    file_split
    .set_index("file_id")["split"]
)

manifest_split_map = (
    sample_manifest
    .groupby(
        "parent_file_id"
    )["split"]
    .first()
)

comparison = (
    expected_split_map
    ==
    manifest_split_map
)

assert comparison.all()

print(
    "PASS: Every sample inherited "
    "the correct SPLIT-001 assignment."
)

# 3E. Validate PREP-001 window length
assert np.allclose(
    sample_manifest[
        "window_duration_s"
    ],
    WINDOW_DURATION_S,
)

assert (
    sample_manifest[
        "source_window_num_samples"
    ]
    ==
    int(
        WINDOW_DURATION_S
        * SOURCE_SAMPLE_RATE_HZ
    )
).all()

assert (
    sample_manifest[
        "expected_model_num_samples"
    ]
    ==
    int(
        WINDOW_DURATION_S
        * TARGET_SAMPLE_RATE_HZ
    )
).all()

print(
    "PASS: Window lengths match PREP-001."
)

# 3F. Check target completeness
target_columns = [
    "rpm_mean",
    "rpm_median",
    "rpm_min",
    "rpm_max",
    "rpm_std",
    "rpm_range",
    "torque_mean_nm",
    "torque_median_nm",
    "torque_min_nm",
    "torque_max_nm",
    "torque_std_nm",
    "torque_range_nm",
]

missing_target_values = (
    sample_manifest[
        target_columns
    ]
    .isna()
    .sum()
)

display(
    missing_target_values
)

assert (
    missing_target_values.sum()
    == 0
)

print(
    "PASS: All target statistics are complete."
)

PASS: All SPLIT-001 source files were processed successfully.
PASS: Every model sample has a unique sample_id.
PASS: Every SPLIT-001 source WAV generated at least one sample.
PASS: No parent WAV spans multiple dataset splits.
PASS: Every sample inherited the correct SPLIT-001 assignment.
PASS: Window lengths match PREP-001.


,0
rpm_mean,0
rpm_median,0
rpm_min,0
rpm_max,0
rpm_std,0
rpm_range,0
torque_mean_nm,0
torque_median_nm,0
torque_min_nm,0
torque_max_nm,0


PASS: All target statistics are complete.
